# 04 - Spark + Apache Iceberg Setup

Mục tiêu:
- Khởi tạo SparkSession
- Kết nối Iceberg REST Catalog
- Kết nối MinIO
- Tạo namespace Bronze / Silver / Gold
- Kiểm tra Spark → Iceberg REST → MinIO


## 4.1. Tạo SparkSession


In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

ICEBERG_VERSION = "1.11.0"
HADOOP_VERSION = "3.4.2"

PACKAGES = (
    f"org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:{ICEBERG_VERSION},"
    f"org.apache.iceberg:iceberg-aws-bundle:{ICEBERG_VERSION},"
    f"org.apache.hadoop:hadoop-aws:{HADOOP_VERSION}"
)

AWS_ACCESS_KEY_ID = (
    os.environ.get("AWS_ACCESS_KEY_ID")
    or os.environ.get("MINIO_ROOT_USER")
)

AWS_SECRET_ACCESS_KEY = (
    os.environ.get("AWS_SECRET_ACCESS_KEY")
    or os.environ.get("MINIO_ROOT_PASSWORD")
)

if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    raise RuntimeError(
        "Không tìm thấy credential MinIO trong biến môi trường."
    )

spark = (
    SparkSession.builder
    .appName("EcommerceLakehouseSetup")
    .master("local[*]")

    .config("spark.jars.packages", PACKAGES)

    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )

    .config(
        "spark.sql.catalog.lakehouse",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config(
        "spark.sql.catalog.lakehouse.type",
        "rest"
    )
    .config(
        "spark.sql.catalog.lakehouse.uri",
        "http://iceberg-rest:8181"
    )

    .config(
        "spark.sql.catalog.lakehouse.io-impl",
        "org.apache.iceberg.aws.s3.S3FileIO"
    )
    .config(
        "spark.sql.catalog.lakehouse.s3.endpoint",
        "http://minio:9000"
    )
    .config(
        "spark.sql.catalog.lakehouse.s3.path-style-access",
        "true"
    )

    .config(
        "spark.hadoop.fs.s3a.endpoint",
        "http://minio:9000"
    )
    .config(
        "spark.hadoop.fs.s3a.endpoint.region",
        "us-east-1"
    )
    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true"
    )
    .config(
        "spark.hadoop.fs.s3a.connection.ssl.enabled",
        "false"
    )
    .config(
        "spark.hadoop.fs.s3a.access.key",
        AWS_ACCESS_KEY_ID
    )
    .config(
        "spark.hadoop.fs.s3a.secret.key",
        AWS_SECRET_ACCESS_KEY
    )
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )

    .getOrCreate()
)

print("✅ SparkSession created")
print("Spark version:", spark.version)


## 4.2. Kiểm tra Catalog


In [ ]:
spark.sql("SHOW NAMESPACES IN lakehouse").show(truncate=False)


## 4.3. Tạo namespaces


In [ ]:
for namespace in ["bronze", "silver", "gold"]:
    spark.sql(
        f"CREATE NAMESPACE IF NOT EXISTS lakehouse.{namespace}"
    )
    print(f"✅ lakehouse.{namespace}")

spark.sql("SHOW NAMESPACES IN lakehouse").show(truncate=False)


## 4.4. Smoke test Iceberg


In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.bronze.connection_test (
    id INT,
    message STRING
)
USING iceberg
""")


In [ ]:
spark.sql("""
INSERT INTO lakehouse.bronze.connection_test
VALUES (1, 'Spark -> Iceberg REST -> MinIO OK')
""")

spark.sql("""
SELECT *
FROM lakehouse.bronze.connection_test
""").show(truncate=False)


In [ ]:
spark.sql("""
SELECT
    committed_at,
    snapshot_id,
    operation
FROM lakehouse.bronze.connection_test.snapshots
ORDER BY committed_at DESC
""").show(truncate=False)


## 4.5. Xóa bảng test


In [ ]:
spark.sql("""
DROP TABLE IF EXISTS lakehouse.bronze.connection_test
""")

print("✅ Đã xóa connection_test")


## 4.6. Hoàn tất


In [ ]:
spark.sql("SHOW NAMESPACES IN lakehouse").show(truncate=False)
print("✅ Step 04 hoàn tất")
